In [3]:
# imports
import numpy as np
import pandas as pd


from collections import defaultdict#, namedtuple
#from itertools import chain
#from math import sqrt
from operator import itemgetter


In [4]:
# bring in intersections -> DataFrame intersections 
intersection_data_path = ("/Users/bencampbell/code/county_coverage/data/cleaner/" +
                          "intersections_data.json")

# import intersection data
def read_in_intersections(path_to_intersection_json):
    out = pd.read_json(path_to_intersection_json)
    # fix some things on import
    out = out.set_index("INTID")
    out.GEOMETRY = out.GEOMETRY.apply(np.array)
    return out             

intersections = read_in_intersections(intersection_data_path)

#intersections.GEOMETRY = intersections.GEOMETRY.apply(np.array)
#intersections.GEOMETRY.iloc[0] # == array([-85.51043903,  38.20587931]) # good 

intersections.head()

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
5710837346,REHL RD,4976,W REHL CT,6856,"[-85.5104390301, 38.2058793146]"
10005800273,REHL RD,4976,TUCKER STATION RD,5908,"[-85.52814980550001, 38.2003487994]"
14300767569,REHL RD,4976,TUCKER STATION RD,5908,"[-85.5284139077, 38.2003530575]"
18011691414,I 64 EAST,3076,I 265 RAMP,8763,"[-85.5049493351, 38.2225961436]"
23945910678,I 265 NORTH,8197,I 265 RAMP,8763,"[-85.5055416176, 38.2221199919]"


In [5]:
# bring in centerlines -> DataFrame centerlines
centerlines_path = "/Users/bencampbell/code/county_coverage/data/cleaner/centerlines_data.json"
centerline_data = pd.read_json(centerlines_path)

# fix geometry column
split_geo = centerline_data.GEOMETRY.transform({"GEOLOW":itemgetter(0), 'GEOHI':itemgetter(-1)})
centerline_data['GEOLOW'] = split_geo.GEOLOW.apply(np.array)
centerline_data['GEOHI'] = split_geo.GEOHI.apply(np.array)

keep_columns = ["ROADNAME", "SIFID", "SIFIDLOW", "SIFIDHI", "GEOLOW", "GEOHI", "CORE_CLASS"]
centerline_data = centerline_data[keep_columns]
#centerlines.head()
#type(centerlines.GEOLOW.iloc[0])


# remove interstates from centerlines ?\
# - and other roads like ramps?
exclusions = ('EXPRESSWAY', 'INTERSTATE RAMP')
excluded_centerlines = centerline_data[centerline_data.CORE_CLASS.isin(exclusions)].index

centerlines = centerline_data[~centerline_data.CORE_CLASS.isin(exclusions)]

# remove these from intersections as well? 

centerlines.head()

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
1,SERENITY CT,8665,1550,8594,"[-85.6809503218, 38.1588670887]","[-85.6812346349, 38.1581804844]",LOCAL
2,S 28TH ST,5926,2854,2470,"[-85.8012012237, 38.230563793]","[-85.8013692698, 38.2293433595]",LOCAL
3,BEECH ST,473,6487,10596,"[-85.8050149882, 38.2289330216]","[-85.8048316249, 38.2275378873]",LOCAL
4,GARDEN DR,2442,13394,4702,"[-85.6802054534, 38.2480671366]","[-85.6798630078, 38.2476081599]",PRIMARY COLLECTOR
5,PARKWAY DR,4573,4162,8594,"[-85.7420397977, 38.2118147709]","[-85.7416475277, 38.2119685968]",LOCAL


In [6]:
def index_by_SIFID_pairs(intersection_sifid_pairs, centerline_sifid_pairs) -> pd.DataFrame:
    return pd.concat((intersection_sifid_pairs, centerline_sifid_pairs), axis=1,
                    # map intersection ids to centerline ids where their sifid pair indexes match
                    # matches are collections of intersection ids/indexes and centerline ids/indexes
                        ).dropna(how='any')
                    # if either collection of ids is empty (or both), there is no match.

# def build_match_dicts(intersection_sifid_index, centerline_sifid_index, 
#                       intersection_dict, centerline_dict) -> None:
#     # build intersection sifid pair / centerline sifid pair index
#     matchset = index_by_SIFID_pairs(intersection_sifid_index, centerline_sifid_index)

#     for _, intersection_ids, centerline_ids in matchset.itertuples():
#         for int_id in intersection_ids:
#             # gather centerline ids into appropriate dictionaries, indexed by intersection ids
#             intersection_dict[int_id].update(centerline_ids)
#             centerline_dict[int_id].update(centerline_ids)


In [7]:
# # old
# # a different tactic -> it works well!
# def build_2(intersection_sifid_index, centerline_sifid_index, 
#                       intersection_dict, centerline_dict) -> None:
#     # build intersection sifid pair / centerline sifid pair index
#     matchset = index_by_SIFID_pairs(intersection_sifid_index, centerline_sifid_index)

#     for _, intersection_ids, centerline_ids in matchset.itertuples():
#         for int_id in intersection_ids:
#             # gather centerline ids into appropriate dictionaries, indexed by intersection ids
#             for cl_id in centerline_ids:
#                 intersection_dict[(int_id, cl_id)] = True
#                 centerline_dict[(int_id, cl_id)] = True


In [8]:



# get groups
def get_groups(df, by) -> dict:
    return pd.Series(df.groupby(by=by).groups)

centerlines_by_SIFID_SIFIDLOW = get_groups(centerlines, ['SIFID', 'SIFIDLOW'])
centerlines_by_SIFID_SIFIDHI = get_groups(centerlines, ['SIFID', 'SIFIDHI'])

intersections_by_FST_SEC = get_groups(intersections, ['FST_SIFID', 'SEC_SIFID'])
# If the index of one of the centerline groups matches an index in this group, that means
#
#   (centerline == centerlines.loc[centerline index])
#   intersection[FST_SIFID] == centerline[SIFID]
#                   and
#   intersections[SEC_SIFID] == either centerline[SIFIDLOW] or centerline[SIFIDHI]
# depending on which set of centerline groups, which we built above, that we are using. 

# The names FST_SIFID / SEC_SIFID -> First / Second intersection SIFID are convention and
# the order is arbitrary. Reversing the order of the index allows us to match SEC_SIFID efficiently.
intersections_by_SEC_FST = get_groups(intersections, ['SEC_SIFID', 'FST_SIFID'])
# If the index of one of the centerline groups is an index in this group, that means
#
#   intersection[SEC_SIFID] == centerline[SIFID]
#                   and
#   intersections[FST_SIFID] == either centerline[SIFIDLOW] or centerline[SIFIDHI]


# # create dictionaries to store future series info
# intersections_FST_match = defaultdict(set)
# intersections_SEC_match = defaultdict(set)
# centerlines_LOW_match = defaultdict(set)
# centerlines_HI_match = defaultdict(set)

# # TODO rename things better

# intm1 = dict()#pd.Series(name='FST_match')
# intm2 = dict()#pd.Series(name='SEC_match')
# clLOWm = dict()#pd.Series(name='SIFIDLOW_match')
# clHIm = dict()#pd.Series(name='SIFIDHI_match')



# build_2(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDLOW, intm1, clLOWm)
# build_2(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDHI, intm1, clHIm)
# build_2(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDLOW, intm2, clLOWm)
# build_2(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDHI, intm2, clHIm)
# # build step takes 30s! # not anymore very fast now for some reaason.
# # TODO rename things better


In [9]:
mapping = defaultdict(int)

start_value = 0b1000 # set start_value flag for first iteration over itersections_by_FST_SEC
for I in (intersections_by_FST_SEC, intersections_by_SEC_FST):

    cvalue = 0b10 # set cvalue flag for first iteration over centerlines_by_SIFID_SIFIDLOW
    for C in (centerlines_by_SIFID_SIFIDLOW, centerlines_by_SIFID_SIFIDHI):
        set_value = start_value + cvalue # combine flags into code
        matchset = index_by_SIFID_pairs(I, C)
        for _, intersection_ids, centerline_ids in matchset.itertuples():
            for intersection_id in intersection_ids:
                for cl_id in centerline_ids:
                    mapping[(intersection_id, cl_id)] |= set_value # set code, preserving flags that are already set
        cvalue = 0b01 # set cvalue flag for next iteration over centerlines_by_SIFID_SIFIDHI

    start_value = 0b0100 # set start_value flag for next iteration over itersections_by_SEC_FST
        
mdf = matchdfnumeric = pd.Series(mapping)

# 
fs= {'fst': lambda x: bool(x & 0b1000),
     'sec': lambda x: bool(x & 0b0100),
     'low': lambda x: bool(x & 0b0010),
     'hi':  lambda x: bool(x & 0b0001)}

match_matrix = mdf.transform(fs).convert_dtypes() # convert the codes to something more easy to use

match_matrix[match_matrix.sum(axis=1) == 2]
match_matrix

fst    sec    low     hi
589991067518338 10250    True  False   True  False
                28655    True  False   True  False
590506463593858 10250    True  False   True  False
                28655    True  False   True  False
409323268213029 26919    True  False   True  False
...                       ...    ...    ...    ...
847134907955010 174788  False   True  False   True
847207302035270 175429  False   True  False   True
847212217628486 175434  False   True  False   True
847246577432390 175435  False   True  False   True
847250872399689 175437  False   True  False   True

[64111 rows x 4 columns]

In [10]:


mapping = defaultdict(int)

fst_mask = 0b0001
sec_mask = 0b0010
low_mask = 0b0100
hi_mask = 0b1000

for imask, I in ((fst_mask, intersections_by_FST_SEC), (sec_mask, intersections_by_SEC_FST)):
    for cmask, C in ((low_mask, centerlines_by_SIFID_SIFIDLOW), (hi_mask, centerlines_by_SIFID_SIFIDHI)):
        matchset = index_by_SIFID_pairs(I, C)
        set_cols = imask + cmask
        for (s1, s2), intersection_ids, centerline_ids in matchset.itertuples():
            for intersection_id in intersection_ids:
                for cl_id in centerline_ids:
                    mapping[(intersection_id, cl_id)] |= set_cols

matchdfnumeric = pd.Series(mapping)

matchdfnumeric.index.set_names(('int_id', 'cl_id'), inplace=True)

In [11]:
# look up roadname by sifid


names = centerlines.groupby("SIFID").ROADNAME.apply(set)
all(names.apply(len) == 1) # -> True. :)
sifid_to_roadname = names.apply(set.pop)

sifid_to_roadname



SIFID
1                    NO NAME
2                   ABBEY RD
3               ABBEYWOOD RD
4           ABBOTTS BEACH RD
5                  ABELL AVE
                ...         
15638    HURSTBOURNE VIEW DR
15639            MEGAWATT DR
15640           GREY WOLF DR
15641        MEADOWSTONE TRL
15642               ASHER CT
Name: ROADNAME, Length: 11330, dtype: object

In [ ]:
iw = matchdfnumeric.index.to_frame()

#int_d = intersections[['FST_ROADNAME', 'SEC_ROADNAME']]
cl_d = centerlines[['ROADNAME', 'SIFIDLOW', 'SIFIDHI']]

def get_int_names(int_id):
    int_d.loc[int_id]

def get_cl_names(cl_id):
    rw, sl, sh = cl_d.loc[cl_id]
    sl = sifid_to_roadname.get(sl)
    sh = sifid_to_roadname.get(sh)
    return pd.Series({
        'ROADNAME': rw,
        'LO_cross': sl,
        'HI_cross': sh})

NAME_df = iw.cl_id.apply(get_cl_names)
#matchdfnumeric.index.to_frame()
#4.9 sec

In [19]:
NAME_df

col_name                                    
                                ROADNAME          LO_cross          HI_cross
int_id          cl_id                                                       
589991067518338 10250            NO NAME      BLUE WING DR              None
                28655            NO NAME      BLUE WING DR              None
590506463593858 10250            NO NAME      BLUE WING DR              None
                28655            NO NAME      BLUE WING DR              None
409323268213029 26919            NO NAME    CAMP GROUND RD              None
...                                  ...               ...               ...
847134907955010 174788   RINGING BELL LN  HALDEN RIDGE WAY  RINGING BELL CIR
847207302035270 175429  COPPER DRIFT WAY  BRICK FORGE PASS     BLACKSMITH RD
847212217628486 175434  COPPER DRIFT WAY              None    PINE BELLOW LN
847246577432390 175435  COPPER DRIFT WAY    PINE BELLOW LN  BRICK FORGE PASS
847250872399689 175437      FIRESCALE CT              None  BRICK FORGE PASS

[64111 rows x 3 columns]

In [19]:
cl_dat = centerlines[['SIFID', 'SIFIDLOW', 'SIFIDHI']]

def gc(ci):
    return cl_dat.loc[ci]

sifm = matchdfnumeric.index.to_frame().cl_id.apply(gc)
sifm


SIFID  SIFIDLOW  SIFIDHI
int_id          cl_id                           
589991067518338 10250       1       591     8594
                28655       1       591     8594
590506463593858 10250       1       591     8594
                28655       1       591     8594
409323268213029 26919       1       897     8594
...                       ...       ...      ...
847134907955010 174788  15593     15254    15594
847207302035270 175429  15597     15599    10281
847212217628486 175434  15597      8594    15598
847246577432390 175435  15597     15598    15599
847250872399689 175437  15600      8594    15599

[64111 rows x 3 columns]

In [21]:

any(matchdfnumeric == 0b0011) # none
any(matchdfnumeric == 0b0000) # false # none of these either

data = {"int_match": matchdfnumeric & 0b0011,
        "cl_sifid": sifm.SIFID,
        "low_match": sifm.SIFIDLOW[(matchdfnumeric & low_mask) != 0],
        "hi_match": sifm.SIFIDHI[(matchdfnumeric & hi_mask) != 0]}


#dd = pd.concat((int_ser, sifm.SIFID, low_ser, hi_ser), axis=1, keys=['int_match', 'cl_sifid', 'low_match', 'hi_match']).convert_dtypes()

SIFdf = pd.DataFrame.from_dict(data).convert_dtypes()

In [1]:
def ix(columns, *labels):
    out = list()
    columns = iter(coluns)
    for label, column in zip(labels, columns):
        out.append((label, column))
    else:
        # default = last value of label
        for column in columns:
            out.append((label, column))
    return pd.MultiIndex.from_tuples(out)

ix(NAME_df.columns, 'hello')

NAME_df.columns=ix(NAME_df.columns, 'hello')

NameError: name 'NAME_df' is not defined

In [8]:

euclidean_distance = np.linalg.norm
#

#
intersections_GEO = intersections.GEOMETRY
centerlines_GEOLOW = centerlines.GEOLOW
centerlines_GEOHI = centerlines.GEOHI


In [ ]:
#ids = mdf.index.to_series().transform({'intersection_ids':itemgetter(0), 'centerline_ids':itemgetter(1)})
ids = mdf.index.to_frame(name=['intersection_ids',"centerline_ids"])

int_points = ids.intersection_ids.apply(lambda intid: intersections_GEO.at[intid])

hi_points = ids.centerline_ids.apply(lambda cl_id: centerlines_GEOHI.at[cl_id])
hi_dist = (int_points - hi_points).apply(euclidean_distance)

low_points = ids.centerline_ids.apply(lambda cl_id: centerlines_GEOLOW.at[cl_id])
low_dist = (int_points - low_points).apply(euclidean_distance)

def encode_closer_point(difference):
    if difference < 0: # hi_dist < low_dist
        return 'hi'
    elif difference > 0: # hi_dist > low_dist
        return 'low'
    elif difference == 0: # hi_dist == low_dist
        return 'both'
    else:
        return pd.NA

dist_diff = (hi_dist - low_dist)
geo_end = dist_diff.apply(encode_closer_point)


geo_end.hasnans # -> False.
# checking something;
(geo_end[geo_end != 'low'] == geo_end[(geo_end == 'hi') | (geo_end == 'both')]).all() # -> True

np.True_

In [10]:
close_points = pd.concat((
    hi_points[geo_end != 'low'], 
    low_points[geo_end == 'low']))

close_distance = pd.concat((
    hi_dist[geo_end != 'low'],
    low_dist[geo_end == 'low']))

far_points = pd.concat((
    hi_points[geo_end == 'low'],
    low_points[geo_end == 'hi']))

far_distance = pd.concat((
    hi_dist[geo_end == 'low'],
    low_dist[geo_end == 'hi']))


working = pd.DataFrame(pd.Series(int_points, name='int_point'))
working['geo_end'] = geo_end
working['close_point'] = close_points
working['far_point'] = far_points
working['close_dist'] = close_distance
working['far_dist'] = far_distance
working['dist_diff'] = dist_diff.abs()

working.index.set_names(('int_id', 'cl_id'), inplace=True)
geo_data = working

geo_data

int_point geo_end  \
int_id          cl_id                                             
589991067518338 10250   [-85.8920292933, 38.1439149601]     low   
                28655   [-85.8920292933, 38.1439149601]     low   
590506463593858 10250      [-85.892052272, 38.14321909]     low   
                28655      [-85.892052272, 38.14321909]     low   
409323268213029 26919   [-85.8301302406, 38.2136024637]     low   
...                                                 ...     ...   
847134907955010 174788  [-85.4966887661, 38.2906992036]      hi   
847207302035270 175429  [-85.5586236853, 38.1263603351]      hi   
847212217628486 175434  [-85.5588442442, 38.1280189286]      hi   
847246577432390 175435  [-85.5585370685, 38.1271574049]      hi   
847250872399689 175437  [-85.5575279734, 38.1270959347]      hi   

                                                 close_point  \
int_id          cl_id                                          
589991067518338 10250         [-85.8920342109, 38.143922224]   
                28655        [-85.8920571895, 38.1432263537]   
590506463593858 10250         [-85.8920342109, 38.143922224]   
                28655        [-85.8920571895, 38.1432263537]   
409323268213029 26919        [-85.8301351454, 38.2136097438]   
...                                                      ...   
847134907955010 174788       [-85.4966935792, 38.2907065141]   
847207302035270 175429       [-85.5586285042, 38.1263676111]   
847212217628486 175434       [-85.5588490633, 38.1280262049]   
847246577432390 175435  [-85.5585418875, 38.127164681000004]   
847250872399689 175437       [-85.5575327922, 38.1271032109]   

                                                   far_point  close_dist  \
int_id          cl_id                                                      
589991067518338 10250        [-85.8901779074, 38.1438753086]    0.000009   
                28655        [-85.8953593354, 38.1433191649]    0.000689   
590506463593858 10250        [-85.8901779074, 38.1438753086]    0.000703   
                28655        [-85.8953593354, 38.1433191649]    0.000009   
409323268213029 26919        [-85.8320107851, 38.2120229119]    0.000009   
...                                                      ...         ...   
847134907955010 174788       [-85.4975897985, 38.2902281308]    0.000009   
847207302035270 175429  [-85.5585418875, 38.127164681000004]    0.000009   
847212217628486 175434       [-85.5589900382, 38.1289572117]    0.000009   
847246577432390 175435       [-85.5588490633, 38.1280262049]    0.000009   
847250872399689 175437       [-85.5566407119, 38.1273321786]    0.000009   

                        far_dist  dist_diff  
int_id          cl_id                        
589991067518338 10250   0.001852   0.001843  
                28655   0.003383   0.002694  
590506463593858 10250   0.001986   0.001283  
                28655   0.003309   0.003300  
409323268213029 26919   0.002456   0.002447  
...                          ...        ...  
847134907955010 174788  0.001017   0.001008  
847207302035270 175429  0.000808   0.000800  
847212217628486 175434  0.000950   0.000941  
847246577432390 175435  0.000923   0.000914  
847250872399689 175437  0.000918   0.000909  

[64111 rows x 7 columns]

In [ ]:

def get_intersection_names(intersection_id):
    in1 = intersections.at[intersection_id, 'FST_ROADNAME']
    in2 = intersections.at[intersection_id, 'SEC_ROADNAME']
    return in1, in2

names = centerlines.groupby("SIFID").ROADNAME.apply(set)#.apply(lambda x:len(x)==1).all() # true
sifid_names = names.apply(lambda x:x.pop())

def roadname_by_sifid(sifid):
    return sifid_names[sifid]

def get_names(int_id, cd_id):
    inames = get_intersection_names(intersection_id)
    cnames = roadname_by_sifid(cl_id)
    return {'intersection':inames, 'centerline':cnames}



'CEDAR BROOK CT'

In [108]:
intids = working.index.to_frame().int_id

def check_names(int_id):
    int_ = intersections.loc[int_id]
    int_fst = int_.FST_SIFID
    int_fst_name = int_.FST_ROADNAME
    cl_sif = sifid_names[int_fst]
    return int_fst_name, cl_sif

intids.apply(check_names)

intersections.loc[intids].FST_SIFID.apply(lambda sifid:sifid_names.get(sifid))
intersections.loc[intids].SEC_SIFID.apply(lambda sifid:sifid_names.get(sifid))

INTID
590467808889140      CANE RUN RD
38521867941168      KENTUCKY AVE
589634891494704     KENTUCKY AVE
706470196341142           KY 841
706470196341142           KY 841
                        ...     
654251992213784      KY-841 RAMP
654251992213784      KY-841 RAMP
2696491761592600     KY-841 RAMP
2696491761592600     KY-841 RAMP
2696491761592600     KY-841 RAMP
Name: SEC_SIFID, Length: 89, dtype: object

In [ ]:
working = geo_data[geo_data.close_dist >= .1]
working

iw = working.index.to_frame()
iw

def get_int_data(int_id):
    return intersections.loc[int_id][["FST_ROADNAME", "FST_SIFID", "SEC_ROADNAME", "SEC_SIFID"]]

def get_cl_data(cl_id):
    return centerlines.loc[cl_id][['ROADNAME', 'SIFID', 'CORE_CLASS']]

data = pd.concat((
iw.cl_id.transform({"centerline_data":get_cl_data}) ,
iw.int_id.transform({'intersection_data':get_int_data})
), axis=1)

data


def mmft(label, df):
    if isinstance(label, str):
        return pd.MultiIndex.from_tuples((label, col) for col in df.columns)
    else:
        return pd.MultiIndex.from_tuples((L, C) for L, C in zip(label, df.columns))
    


def dd(geodf):
    #ex = geodf[['int_point
    #out = list()
    iw = geodf.index.to_frame()
    cl_geo = geodf[['int_point', 'close_point', 'close_dist', 'geo_end']]
    cl_geo.columns = mmft(('int_geo', 'cl_geo', 'cl_geo', 'cl_geo'), cl_geo)

    cl_data = iw.cl_id.apply(get_cl_data)
    cl_data.columns = mmft('cl_data', cl_data)
    # TODO get hi/low sifid / roadname

    int_data = iw.int_id.apply(get_int_data)
    int_data.columns = mmft("int_data", int_data)

    out = [int_data, cl_geo, cl_data]
    return pd.concat(out, axis=1)
    
    int_data = iw.int_id.apply(get_int_data)
    int_info = pd.concat((int_data, geodf.int_point), axis=1)
    int_info.columns = pd.MultiIndex.from_tuples(('int_data', col) for col in int_info.columns)
    
    return pd.concat((int_info, cl_info), axis=1).sort_index()

#    pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
#c.columns = pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
#c

see = dd(working)

#see[~see.int_data.FST_ROADNAME.str.contains('841')]

see

Index(['ROADNAME', 'SIFID', 'SIFIDLOW', 'SIFIDHI', 'GEOLOW', 'GEOHI',
       'CORE_CLASS'],
      dtype='object')

In [ ]:
iw = working.index.to_frame()

def get_mm(iw):
    cl_id = iw.cl_id
    cl = centerlines.loc[cl_id]
    int_id = iw.int_id
    m = match_matrix.loc[int_id]
    if m.FST:
        ...

    cl_lowsif = cl_hisif = None
    if m.low:
        cl_lowsif = cl.SIFIDLOW
    if m.hi:
        cl_hisif = cl.SIFIDHI

def f(row):
    return type(row)

match_matrix

fst    sec    low     hi
589991067518338 10250    True  False   True  False
                28655    True  False   True  False
590506463593858 10250    True  False   True  False
                28655    True  False   True  False
409323268213029 26919    True  False   True  False
...                       ...    ...    ...    ...
847134907955010 174788  False   True  False   True
847207302035270 175429  False   True  False   True
847212217628486 175434  False   True  False   True
847246577432390 175435  False   True  False   True
847250872399689 175437  False   True  False   True

[64111 rows x 4 columns]

In [94]:
match_matrix.loc[847212217628486, 175434].loc[['low', 'hi']]

low    False
hi      True
Name: (847212217628486, 175434), dtype: boolean

In [48]:

c = iw.cl_id.apply(get_cl_data)
i = iw.int_id.apply(get_int_data)


df=pd.DataFrame({'a':[1,2,3],'b':[4,5,6]})

columns=[('c','a'),('c','b')]

df.columns=pd.MultiIndex.from_tuples(columns)
df

pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
c.columns = pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
c

cl_data                       
                            ROADNAME  SIFID      CORE_CLASS
int_id           cl_id                                     
590467808889140  29463       NO NAME      1           LOCAL
38521867941168   6218   COLUMBIA AVE   1245           LOCAL
589634891494704  21710  COLUMBIA AVE   1245           LOCAL
706470196341142  903     KY-841 RAMP  14021  MAJOR ARTERIAL
                 18413   KY-841 RAMP  14021  MAJOR ARTERIAL
...                              ...    ...             ...
654251992213784  6590    KY-841 RAMP  14021  MAJOR ARTERIAL
                 7706    KY-841 RAMP  14021  MAJOR ARTERIAL
2696491761592600 6590    KY-841 RAMP  14021  MAJOR ARTERIAL
                 7706    KY-841 RAMP  14021  MAJOR ARTERIAL
                 11399   KY-841 RAMP  14021  MAJOR ARTERIAL

[89 rows x 3 columns]

In [15]:
s= working.index.get_level_values('cl_id')

centerlines.loc[s].ROADNAME.value_counts()

ROADNAME
KY-841 RAMP     39
KY 841          36
COLUMBIA AVE     4
CANE RUN RD      4
KENTUCKY AVE     4
NO NAME          2
Name: count, dtype: int64

In [16]:
mi = intersections.loc[working.index.get_level_values('int_id')]

# KY 841 / I 265 / Watterson Expwy and ramps
mi[(mi.FST_ROADNAME.str.contains("841") & mi.SEC_ROADNAME.str.contains("841"))]

# other weird matches
wm = mi[~(mi.FST_ROADNAME.str.contains("841") & mi.SEC_ROADNAME.str.contains("841"))]

working.loc[wm.index, :]
wm
mi


,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
int_id,,,,,
590467808889140,NO STREET NAME,1,CANE RUN RD,906,"[-85.8967398176, 38.1433580312]"
38521867941168,COLUMBIA AVE,1245,KENTUCKY AVE,3356,"[-85.6184455169, 38.2600006953]"
589634891494704,COLUMBIA AVE,1245,KENTUCKY AVE,3356,"[-85.8480563282, 38.1508005158]"
706470196341142,KY-841 RAMP,14021,KY 841,14195,"[-85.7024627213, 38.1152264972]"
706470196341142,KY-841 RAMP,14021,KY 841,14195,"[-85.7024627213, 38.1152264972]"
...,...,...,...,...,...
654251992213784,KY 841,14195,KY-841 RAMP,14021,"[-85.8700263878, 38.091373397]"
654251992213784,KY 841,14195,KY-841 RAMP,14021,"[-85.8700263878, 38.091373397]"
2696491761592600,KY 841,14195,KY-841 RAMP,14021,"[-85.8769126885, 38.0929399762]"


In [ ]:


    

#display(
#info[['int_point', 'close_point', 'close_dist', 'geo_end', ]])



wmm = data[~data.intersection_data.FST_ROADNAME.str.contains('841')]

display(
working.loc[wmm.index][['int_point', 'close_point', 'close_dist', 'geo_end', ]],
wmm)


NameError: name 'w' is not defined

In [ ]:
centerlines[centerlines.ROADNAME.str.contains("KENTUCKY AVE")]
centerlines[(centerlines.SIFID == 3356) & ((centerlines.SIFIDLOW == 1245) | (centerlines.SIFIDHI == 1245))]

#wm = wm.groupby(by="FST_SIFID").apply(lambda x:x)


,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
2474,KENTUCKY AVE,3356,1245,4725,"[-85.848070012, 38.1508080266]","[-85.848221191, 38.1490259808]",LOCAL
5523,KENTUCKY AVE,3356,2249,1245,"[-85.6148949649, 38.2557432145]","[-85.6184503633, 38.2600079942]",LOCAL
10465,KENTUCKY AVE,3356,8594,1245,"[-85.8478987894, 38.1526162695]","[-85.848070012, 38.1508080266]",LOCAL
29055,KENTUCKY AVE,3356,1245,8594,"[-85.6184503633, 38.2600079942]","[-85.620123247, 38.2599229012]",LOCAL


In [245]:
def convert_geo(geo):
    if len(geo) == 2:
        long, lat = geo
        return (lat, long)
    else:
        return [(lat, long) for long, lat in geo]

def swap_point(point):
    x, y = point
    return (y, x)

swap_point((1,2))
        

(2, 1)

In [66]:
# working = mdf.index.to_series()


# def get_geo(row):
#     int_id, cl_id = row
#     out = dict() # tried Series. Took too long. Dict is very fast ( .4 sec)
#     out['intx_geo'] = intx_geo = intersections_GEO.at[int_id]
#     out['cl_geo_low'] = centerlines_GEOLOW.at[cl_id]
#     out['cl_geo_hi'] = centerlines_GEOHI.at[cl_id]
#     return out

# working = pd.DataFrame(mdf.index.to_series().apply(get_geo).to_list(), index=mdf.index)
# hi_dist = (working.intx_geo - working.cl_geo_hi).apply(euclidean_distance)
# low_dist = (working.intx_geo - working.cl_geo_low).apply(euclidean_distance)

# def find_close_closer_point(x):
#     if x < 0:
#         return 'hi'
#     elif x > 0:
#         return 'low'
#     elif x == 0:
#         return 'both'
#     else:
#         return pd.NA

# working['geo_end'] = (hi_dist - low_dist).apply(find_close_closer_point)
# working

# def gp(row):
#     code = row.geo_end
#     if code == 'low':
#         return (row.cl_geo_low, row.cl_geo_hi)
#     elif code == 'hi':
#         return (row.cl_geo_hi,row.cl_geo_low)
#     elif code == 'both':
#         return (row.cl_geo_hi, None)
#     else:
#         return (None, None)

# ee = pd.DataFrame(working.apply(gp, axis=1).to_list(), index=working.index, columns=['close_point', 'far_point'])
# pd.concat((working, ee), axis=1)


In [ ]:
# old versions of code
#     lowdist = euclidean_distance(intx_geo - cl_geo_low)
#     hidist = euclidean_distance(intx_geo - cl_geo_hi)
#     if lowdist < hidist:
#         geo_close = 'low'
#         closer_point = cl_geo_low
#         farther_point = cl_geo_hi
#         close_dist = lowdist
#         far_dist = hidist
#     elif lowdist > hidist:
#         geo_close = 'hi'
#         closer_point = cl_geo_hi
#         farther_point = cl_geo_low
#         close_dist = lowdist
#         far_dist = hidist
#     elif lowdist == hidist:
#         geo_close = 'both'
#         #assert cl_geo_low == cl_geo_hi
#         closer_point = cl_geo_low
#         farther_point = pd.NA
#         close_dist = lowdist
#         far_dist = hidist

#     out['geo_close'] = geo_close
#     out['close_point'] = closer_point
#     out['far_point'] = farther_point
#     out['close_dist'] = close_dist
#     out['far_dist'] = far_dist

#     return out


# working = pd.DataFrame(mdf.index.to_series().apply(find_close_closer_point).to_list(), index=mdf.index)
# #centerlines.loc[working[working.closer_code == 'both'].index.get_level_values(1)] # closer_code = 'both'


In [116]:

diffs = (working['close_dist'] - working['far_dist']).abs()
# some values are zero
# drop these to make finding min easier
dd = diffs.drop(diffs[diffs == 0].index)


dd.min()

np.float64(6.673505504779673e-07)

In [ ]:

working['dist_diff'] = (working.lowdist - working.hidist).abs()

def t(lo, hi):
    if lo < hi:
        return 'low'
    elif lo > hi:
        return 'hi'
    elif lo == hi:
        return 'both'
    else:
        return None

working['closer_point'] = working.lowdist.combine(working.hidist, t)

def gp(row):
    if row.closer_point == 'low':
        return row.cl_geo_low
    elif row.closer_point == 'hi':
        return row.cl_geo_hi

working['closest_point'] = working.apply(gp, axis=1)
working

In [173]:
intm1 = pd.Series(intm1, name='FST_match', dtype='boolean')
intm2 = pd.Series(intm2, name='SEC_match', dtype='boolean')
clLOWm = pd.Series(clHIm, name='SIFIDLOW_match', dtype='boolean')
clHIm = pd.Series(clLOWm, name='SIFIDHI_match', dtype='boolean')

newmatch = pd.concat((intm1, intm2, clLOWm, clHIm), axis=1)
newmatch[newmatch.SIFIDHI_match & newmatch.SIFIDLOW_match]

,,FST_match,SEC_match,SIFIDLOW_match,SIFIDHI_match
323827650687044,18779,True,<NA>,True,True
323879190294596,18779,True,<NA>,True,True
40699115667584,13360,True,<NA>,True,True
40922453966976,13360,True,<NA>,True,True
319438198302071,22602,True,<NA>,True,True
...,...,...,...,...,...
847134907955010,174788,<NA>,True,True,True
847207302035270,175429,<NA>,True,True,True
847212217628486,175434,<NA>,True,True,True
847246577432390,175435,<NA>,True,True,True


In [174]:
intersections.loc[319438198302071]
centerlines.loc[22602]

ROADNAME                           ALFORD AVE
SIFID                                      51
SIFIDLOW                                 5933
SIFIDHI                                  5933
GEOLOW         (-85.797822739, 38.2653975559)
GEOHI         (-85.7985983259, 38.2654832285)
CORE_CLASS                              LOCAL
Name: 22602, dtype: object

In [108]:
def g(f, s):
    #f, s = row
    if f:
        return False
    else:
        True


newmatch[newmatch['FST_match'] & newmatch['SEC_match']] #empty
newmatch[~(newmatch['FST_match'] & newmatch['SEC_match'])] #empty

(newmatch.FST_match.notna().apply(lambda x: (0b1000 if x else 0b0100)) +
newmatch.SIFIDLOW_match.notna().apply(lambda x:(0b0010 if x else 0)) +
newmatch.SIFIDHI_match.notna().apply(lambda x: (0b0001 if x else 0)))




589991067518338  10250     8
                 28655     8
590506463593858  10250     8
                 28655     8
409323268213029  26919     8
                          ..
847134907955010  174788    7
847207302035270  175429    7
847212217628486  175434    7
847246577432390  175435    7
847250872399689  175437    7
Length: 64111, dtype: int64

In [ ]:

# Create indexes, sort information into the appropriate dictionaries.
build_match_dicts(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDLOW, intersections_FST_match, centerlines_LOW_match)
    # Where intersections[FST_SIFID] == centerlines[SIFID] & intersections[SEC_SIFID] == centerlines[SIFIDLOW]
    # Map intersection ids to the centerline ids. Next, do the same for all the other combinations of columns:
build_match_dicts(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDHI, intersections_FST_match, centerlines_HI_match)
build_match_dicts(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDLOW, intersections_SEC_match, centerlines_LOW_match)
build_match_dicts(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDHI, intersections_SEC_match, centerlines_HI_match)

# Convert the dictionaries to Series.
intersections_FST_match = pd.Series(intersections_FST_match, name='FST_match')
intersections_SEC_match = pd.Series(intersections_SEC_match, name='SEC_match')
centerlines_LOW_match = pd.Series(centerlines_LOW_match, name='SIFIDLOW_match')
centerlines_HI_match = pd.Series(centerlines_HI_match, name='SIFIDHI_match')

# Create dataframe by concatenating Series. Intersection id's are the index. 
matchdf = pd.concat((intersections_FST_match, intersections_SEC_match, centerlines_LOW_match, centerlines_HI_match), axis=1)
matchdf.info()

In [413]:
# # # some tests

# #matchdf.apply(lambda x:type(x), axis=1) # -> Series
# #matchdf.apply(lambda x:len(x.values), axis=1) # == 4

# estNA = matchdf.isna().sum(axis=1)
#    # count of null values for each row

# estNA[estNA == 0] # 17887
# estNA[estNA == 1] #   886
# estNA[estNA == 2] #   965
# estNA[estNA > 2]  # empty!


In [414]:
"""Find distance from an intersection point to each possible centerline that it has been mapped to. 
Each centerline has a geometry object that is a list of points. If a centerline connects to an
intersection, the intersetion point should be close to one end of the centerline's geometry or the other."""

euclidean_distance = np.linalg.norm
# norm of a difference vector ([A, A'] - [B, B']) -> Pythagorean theorem
# sqrt((A - B)**2 + (A' - B')**2) = length of the vector as a scalar: distance between two points.

intersections_GEO = intersections.GEOMETRY.apply(np.array)
# converting point to np.array alows easier pointwise math like pointA - pointB where pointA = (a, b) pointB = (y, z)
# without this, we would have to unpack / repack a point every time we want to calculate something based on the 
# difference between coordinates, like this: 
#   a, b = pointA
#   y, z = pointB
#   newPoint = (a-y, b-x)
#   calculation_that_takes_a_point(newPoint)

# it may not be necessary to convert these to np.array. If pointA is an np.array, and operation like 
#       pointA - (x, y) will deal with the second object automatically, returning another np.array.
# But, it is helpful to isolate GEOLOW and GEOHI, and doing the conversion changes nothing mathematically.
# Maybe it's a tiny bit faster ¯\_(ツ)_/¯
centerlines_GEOLOW = centerlines.GEOLOW.apply(np.array)
centerlines_GEOHI = centerlines.GEOHI.apply(np.array)

def combine_sets(row):
    """Union of all sets in a row. Ignore null values."""
    all_ids = set()
    for id_set in row:
        if pd.isna(id_set):
            continue
        else:
            all_ids.update(id_set)
    return all_ids

# aggregate each row
aggregate_centerline_ids = matchdf.apply(combine_sets, axis=1)

def get_distances(row):
    int_point = intersections_GEO.at[row.name]
    distances = dict()
    for cl_id in row.item(): # row.item() is an array of sets and possibly NAs
        low_distance = euclidean_distance(int_point - centerlines_GEOHI.at[cl_id])
        hi_distance = euclidean_distance(int_point - centerlines_GEOLOW.at[cl_id])
        if low_distance < hi_distance:
            match = (low_distance, "GEOLOW")
        elif hi_distance < low_distance:
            match = (hi_distance, "GEOHI")
        elif low_distance == hi_distance:
            # possible because some centerlines represent loops
            match = (hi_distance, "BOTH")
        else: 
            # something weird is going on. I don't think this is actually possible unless there is
            # missing centerline data. This path shouldn't ever run unless np.linalg.norm returns a 
            # weird value in some situation I don't know about.
            match = float('nan') 
        distances[cl_id] = match
    return distances

distance_map = pd.DataFrame(aggregate_centerline_ids).apply(get_distances, axis=1)
# aggregate_centerline_ids has to be cast to a DataFrame first to make .apply work conveniently.

#display(
#distance_map.head(5),
#distance_map.iloc[4]
#)

In [415]:

def filter_id_set(set1, set2):
    if pd.isna(set1) or pd.isna(set2):
        return set1
    else:
        return set1.intersection(set2)

def exclude_distances_by_threshold(df, distance_map, threshold = 1.0):

    def select_keys(distance_dictionary):
        """for each value in a dictionary, keep the key, value pairs where value is below the threshold"""
        filtered_keys = {cl_id for cl_id, value in distance_dictionary.items() if (value[0] < threshold)}
        if filtered_keys:
            return filtered_keys
        else:
            return pd.NA

    #filtered_dmap = distance_map.apply(select)
    #keep = filtered_dmap.apply(lambda D:(D.keys() if D else pd.NA))

    # ids of centerlines that are close enough to keep
    close_enough = distance_map.apply(select_keys)

    data_columns = df[["FST_match", "SEC_match", "SIFIDLOW_match", "SIFIDHI_match"]]
    filtered_df = data_columns.apply(lambda column:column.combine(close_enough, filter_id_set))
    return filtered_df


ex_df = exclude_distances_by_threshold(matchdf, distance_map, 0.1)
ex_df.head()
#matchdf       
    

#distance_map_filtered = distance_map.apply(exclude_by_upper_threshold(.1))
#memo2[memo2.apply(len) >= 1] 
#distance_map_filtered

#keep = distance_map_filtered.apply(lambda D:(D if pd.isna(D) else D.keys()))



#matchdf[["FST_match", "SEC_match", "SIFIDLOW_match", "SIFIDHI_match"]].apply(lambda col:col.combine(keep, k))

,FST_match,SEC_match,SIFIDLOW_match,SIFIDHI_match
589991067518338,"{6313, 10250, 28655, 15159}","{1332, 2389, 21390}","{10250, 1332, 2389, 28655}","{6313, 2389, 21390, 15159}"
590506463593858,"{6313, 10250, 28655, 15159}","{1332, 2389, 21390}","{10250, 1332, 2389, 28655}","{6313, 2389, 21390, 15159}"
409323268213029,{26919},"{24675, 13587}","{24675, 26919}",{13587}
352187318274356,{29463},"{5361, 26722}","{26722, 29463}",{5361}
590467808889140,{31519},"{11220, 32589}",{11220},"{32589, 31519}"


In [416]:


def apply_function_by_distance_key(df, function, *, distance_keys=distance_map, post_function=None, **kwargs):
    if post_function is None:
        post_function = lambda a: a
    
    def get_result(ids, distances):
        if pd.isna(distances):
            return pd.NA
        try:
            if pd.isna(ids):
                return pd.NA
        except ValueError:
            pass
        return post_function(function(ids, key=distances.get, **kwargs))
    
    data_columns = df[["FST_match", "SEC_match", "SIFIDLOW_match", "SIFIDHI_match"]]
    return data_columns.apply(lambda col:col.combine(distance_keys, get_result))

# all to do this:
# (apply_function_by_distance_key(matchdf, sorted).isna() == matchdf.isna()).all().all() # nice, all centerline sets covered and sorted:


matchdf_sorted = apply_function_by_distance_key(matchdf, sorted)

In [417]:
#sec_low_only = matchdf_sorted[matchdf.FST_match.isna() & matchdf.SIFIDHI_match.isna()]
#(sec_low_only.SEC_match == sec_low_only.SIFIDLOW_match).all() # neat they all match
# is this true of similar combinations?
#(sec_low_only.SEC_match == sec_low_only.SIFIDLOW_match).all()

i1, i2 = "FST_match", "SEC_match"
c1, c2 = 'SIFIDLOW_match', 'SIFIDHI_match'

# df = matchdf_sorted
# for i in (i1, i2):
#     for c in (c1, c2):
#         m = df[df[i].isna() & df[c].isna()]
#         if i == i1:
#             i = i2
#         elif i == i2:
#             i = i1
#         if c == c1:
#             c = c2
#         elif c == c2:
#             c = c1
#         print(j, d)
#         display(m)
#         print((m[i] == m[c]).all())
    # all true

counts = matchdf_sorted.isna().sum(axis=1)
matchdf_sorted[counts==2]

matchdf_sorted[matchdf_sorted.SIFIDLOW_match.isna() & matchdf_sorted.SIFIDHI_match.isna()] # no items
matchdf_sorted[matchdf_sorted.FST_match.isna() & matchdf_sorted.SEC_match.isna()] # no items


counts = matchdf_sorted.isna().sum(axis=1)

def t(row):
    row = row.dropna()
    I, C = row.index
    A, B = row.values
    if A == B:
        return I, C, A[0]
    else:
        raise ValueError
    
def gn(row):
    ...

TWOS = matchdf_sorted[counts==2]

pd.DataFrame.from_dict(matchdf_sorted[counts==2].apply(t, axis=1).to_dict()).T

matchdf_sorted[counts==2].apply(lambda col:col.notna().apply(lambda x:col.name if x else pd.NA))

def binary(row):
    out = 0 
    for bit in row:
        out <<= 1
        if bit:
            out += 1
    return f"{out:04b}"

def gv(row):
    A, B = row.dropna()
    if A == B:
        return B
    else:
        raise ValueError
        

TWOS2 = pd.concat((
    TWOS.isna().apply(binary,axis=1),
    TWOS.apply(gv, axis=1)),
        axis = 1, keys=['fields', 'id_values'])


In [ ]:

#TWOS2[TWOS2.id_values.apply(len) > 1]
intersections_GEO.loc[TWOS2.index] 
TWOS2['int_GEO'] = intersections_GEO.loc[TWOS2.index] 
TWOS2

def find_close_closer_point(row):
    out = list()
    for id in row:
        out.append((centerlines.at[id, "GEOLOW"],centerlines.at[id, 'GEOHI']))
    return out

def do2(xs, dd):
    return [dd.get(x) for x in xs]

#TWOS2.id_values.combine(distance_map, do2)

distance_map
distance_map.loc[TWOS2.index]


TWOS2['id_dist'] = TWOS2.id_values.combine(distance_map.loc[TWOS2.index], do2)

def do3(ids, xs):
    for id, geo in zip(ids, xs):
        _, col = geo
        point = centerlines.at[id, col]
    return point

TWOS2['cl_points'] = TWOS2.id_values.combine(TWOS2.id_dist, do3)

TWOS2
#centerlines.GEOHI



,fields,id_values,int_GEO,id_dist,cl_points
583187839915417,0101,[11546],"[-85.8342317183, 38.1884412547]","[(8.773669512807687e-06, GEOHI)]","(-85.8359780841, 38.1888949696)"
415001216295459,0101,[28936],"[-85.7614118967, 38.1903882623]","[(8.764992734092423e-06, GEOHI)]","(-85.7598311197, 38.190408344)"
324570680025475,0101,[9689],"[-85.7249378309, 38.2573126351]","[(8.773923802813555e-06, GEOHI)]","(-85.7253164789, 38.2577592595)"
3401617397254,0101,[140221],"[-85.5184493923, 38.2435007585]","[(8.745700255962354e-06, GEOHI)]","(-85.5180533525, 38.2447271073)"
41240281569111,0101,[27221],"[-85.6276103923, 38.2519695131]","[(8.76086193606409e-06, GEOHI)]","(-85.6266248071, 38.2528013967)"
...,...,...,...,...,...
167762364429688,1010,[24034],"[-85.6480379283, 38.1164473608]","[(0.0010809940750718667, GEOLOW)]","(-85.6518619888, 38.1158760582)"
706248184132984,1010,[24034],"[-85.6491125882, 38.1163833616]","[(8.73613994908614e-06, GEOLOW)]","(-85.6518619888, 38.1158760582)"
695512044398736,1010,[154917],"[-85.6596917485, 38.1542946877]","[(8.745125951554675e-06, GEOLOW)]","(-85.6610839288, 38.1544613325)"
396182080320003,1010,[7798],"[-85.7679352849, 38.2493446865]","[(8.77761006648939e-06, GEOLOW)]","(-85.7678133448, 38.2500429248)"


In [468]:


# TODO
# Build cl/int multiindex df where
#           fst sec low hi geolow geo hi
# [i, c]    bools .....

TWOS.notna().map(int)

def gv(row):
    A, B = row.dropna()
    if A == B:
        return B
    else:
        raise ValueError
    
memo = dict()
for index, ids in TWOS.apply(gv, axis=1).items():
    for id in ids:
        distance, label = distance_map.loc[index].get(id)
        cl_point = centerlines.at[id, label]
        int_point = intersections.at[index, 'GEOMETRY']
        memo[(index, id)] = (int_point, cl_point, distance, label)

pd.DataFrame.from_dict(memo, orient='index')

mx = pd.MultiIndex.from_tuples(memo.keys())

#pd.DataFrame(memo, index=mx)

(
    pd.DataFrame(pd.Series(memo).to_list(), index=mx,
                 columns = ['int_point', 'cl_point', 'distance', 'cl_label']))
        



,,int_point,cl_point,distance,cl_label
583187839915417,11546,"(-85.8342317183, 38.1884412547)","(-85.8359780841, 38.1888949696)",0.000009,GEOHI
415001216295459,28936,"(-85.7614118967, 38.1903882623)","(-85.7598311197, 38.190408344)",0.000009,GEOHI
324570680025475,9689,"(-85.7249378309, 38.2573126351)","(-85.7253164789, 38.2577592595)",0.000009,GEOHI
3401617397254,140221,"(-85.5184493923, 38.2435007585)","(-85.5180533525, 38.2447271073)",0.000009,GEOHI
41240281569111,27221,"(-85.6276103923, 38.2519695131)","(-85.6266248071, 38.2528013967)",0.000009,GEOHI
...,...,...,...,...,...
167762364429688,24034,"(-85.6480379283, 38.1164473608)","(-85.6518619888, 38.1158760582)",0.001081,GEOLOW
706248184132984,24034,"(-85.6491125882, 38.1163833616)","(-85.6518619888, 38.1158760582)",0.000009,GEOLOW
695512044398736,154917,"(-85.6596917485, 38.1542946877)","(-85.6610839288, 38.1544613325)",0.000009,GEOLOW
396182080320003,7798,"(-85.7679352849, 38.2493446865)","(-85.7678133448, 38.2500429248)",0.000009,GEOLOW


In [377]:


# # M12 = pd.concat((intersections_by_sifid_12,lowm),axis=1).dropna(how='any')
# # # match FST_SIFID in intersection, SIFID/SIFIDLOW in centerlines
# # # reverse index order for lowm to find:
# # # SEC_SIFID matches for intersections, SIFID/SIFIDLOW info for centerlines
# # M21 = pd.concat((intersections_by_sifid_21,lowm),axis=1).dropna(how='any')
# # # note that index for this result it swapped
# # # SEC_SIFID value is first, FIST_SIFID value is second for each intersection

# # intersections_FST_match = defaultdict(set)
# # intersections_SEC_match = defaultdict(set)
# # centerlines_LOW_match = defaultdict(set)

# # set FST_SIFID matches
# for _, intersection_ids, centerline_ids in M12.itertuples():
#     for intersection_id in intersection_ids:
#         intersections_FST_SEC_match[intersection_id].update(centerline_ids)
#     for centerline_id in centerline_ids:
#         centerlines_LOW_match[centerline_id].update(intersection_ids)




# # set SEC_SIFID matches
# for _, intersection_ids, centerline_ids in M21.itertuples():
#     for intersection_id in intersection_ids:
#         intersections_SEC_match[intersection_id].update(centerline_ids)
#     for centerline_id in centerline_ids:
#         centerlines_LOW_match[centerline_id].update(intersection_ids)

# centerlines_LOW_match = pd.Series(centerlines_LOW_match)
# intersections_FST_SEC_match = pd.Series(intersections_FST_SEC_match)
# intersections_SEC_FST_match = pd.Series(intersections_SEC_match)

In [378]:

def get_pair(sifid:int, x:int) -> tuple:
    """create standard SIFID pairs: the lowest SIFID numerically is always first"""
    if x < sifid:
        return (x, sifid)
    else:
        return (sifid, x)
    

def map_intersection_centerline_cross_pairs(intersections, centerlines):
    # get centerline SIFID, SIFIDLOW and SIFID, SIFIDHI pairs mapped to centerline OBJECTID
    memo = defaultdict(set)
    low_pairs = centerlines.SIFID.combine(centerlines.SIFIDLOW, get_pair)
    hi_pairs = centerlines.SIFID.combine(centerlines.SIFIDHI, get_pair)
    for pairset in (low_pairs, hi_pairs):
        for centerline_index, pair in pairset.items():
            memo[pair].add(centerline_index) 
    centerline_pairs = pd.Series(memo).apply(list)


    # get intersection FST_SIFID, SEC_SIFID pairs mapped to intersection id
    memo = defaultdict(set)
    sifid_pairs = intersections['FST_SIFID'].combine(intersections['SEC_SIFID'], get_pair)
    for index, pair in sifid_pairs.items():
        memo[pair].add(index)
    intersection_pairs = pd.Series(memo).apply(list)

    return pd.concat((intersection_pairs, centerline_pairs), axis=1, 
                     keys=('intersection_ids', 'centerline_ids'))

IXCpairs = map_intersection_centerline_cross_pairs(intersections, centerlines)
IXCpairs


intersection_ids  \
4976 6856                                        [5710837346]   
     5908                          [14300767569, 10005800273]   
3076 8763                          [35191560598, 18011691414]   
8197 8763   [143102191442326, 143115076344214, 29093067080...   
3076 8764   [696349919490452, 722862752608660, 72633308618...   
...                                                       ...   
4900 8594                                                 NaN   
7026 15638                                                NaN   
2072 8594                                                 NaN   
8594 15641                                                NaN   
     15642                                                NaN   

                               centerline_ids  
4976 6856               [20353, 52803, 20860]  
     5908   [13504, 7109, 84838, 31468, 9233]  
3076 8763                                 NaN  
8197 8763                                 NaN  
3076 8764                                 NaN  
...                                       ...  
4900 8594                            [179906]  
7026 15638                           [179908]  
2072 8594                            [180228]  
8594 15641                           [180229]  
     15642                           [180236]  

[26868 rows x 2 columns]

In [379]:
match = (IXCpairs.intersection_ids.notna() & IXCpairs.centerline_ids.notna())

IXCno_match = IXCpairs[~match]
IXCmatches = IXCpairs[match]

IXCmatches[IXCmatches.intersection_ids.apply(len) == 1]

intersection_ids                    centerline_ids
4976  6856        [5710837346]             [20353, 52803, 20860]
7631  8704       [79566575913]             [26604, 90022, 23911]
      10427      [83861558290]                     [26604, 4511]
4976  8704       [87315224873]                    [90022, 90023]
1838  7500       [94897276198]             [89993, 15170, 15118]
...                        ...                               ...
4946  15601  [847255167849521]          [175440, 175438, 175439]
10583 15602  [847259462915456]          [175745, 175746, 175747]
4674  6042   [847261337735316]                          [175749]
6232  6810   [847274257577475]  [175752, 175756, 175750, 175751]
7097  15603  [847272347859222]          [175753, 175754, 175755]

[17455 rows x 2 columns]

In [380]:
# old IXP CXP code

# # get centerline SIFID : SIFID HI/LOW pairs

# def get_centerline_pairs(centerlines=centerlines):
#     pairs = defaultdict(set)
#     low_pairs = centerlines.SIFID.combine(centerlines.SIFIDLOW, get_pair)
#     hi_pairs = centerlines.SIFID.combine(centerlines.SIFIDHI, get_pair)
#     for pairset in (low_pairs, hi_pairs):
#         for centerline_index, pair in pairset.items():
#             pairs[pair].add(centerline_index) 
#     return pd.Series(pairs, name='centerline_sifid_pairs')

# CXP = get_centerline_pairs()
# #CXP

# def get_intersection_cross_pairs(intersections=intersections):
#     pairs = defaultdict(set)
#     sifid_pairs = intersections['FST_SIFID'].combine(intersections['SEC_SIFID'], get_pair)
#     for index, pair in sifid_pairs.items():
#         pairs[pair].add(index)
#     return pd.Series(pairs, name='intersection_sifid_pairs')

# IXP = get_intersection_cross_pairs()
# #IXP

# CXP_keys = set(CXP.keys())
# IXP_keys = set(IXP.keys())

# display(CXP, IXP)

In [381]:
#ri = pd.DataFrame.from_dict(roadway_intersections, orient='index')
#ri['intersection_1'] = ri.intersection_1.apply(lambda x:tuple(x) if x else pd.NA)
#ri['intersection_2'] = ri.intersection_2.apply(lambda x:tuple(x) if x else pd.NA)

#ri
#display(intersection_roadways,roadway_intersections)

In [382]:
# # keep: deals with unmatched intersections/centerlines

# # can map
# #ixp_keys & cxp_keys

# # intersections without centerline match:
# #ixp_keys - cxp_keys

# # missing itersection keys
# missing_intersections = set()
# for key in IXP_keys - CXP_keys:
#     missing_intersections.update(IXP[key])
# missing_intersections = list(missing_intersections)

# #
# # centerlines without intersection match:
# #cxp_keys - ixp_keys

# # missing centerline keys
# missing_centerlines = set()
# for key in CXP_keys - IXP_keys:
#     missing_centerlines.update(CXP[key])
# missing_centerlines = list(missing_centerlines)

# #
# # In both cases, attempt to match one or the other sifids, not both.
# # Acommodates one ways, loops and other structures.  


In [383]:
# # CXP = get_centerline_pairs()
# # IXP = get_intersection_cross_pairs()
# def map_intersection_centerline_cross_pairs(intersection_cross_pairs=IXP,
#                                              centerline_cross_pairs=CXP,
#                                              reverse=False):
    
#     matches = list()
    
#     # easy matches
#     keys = IXP_keys & CXP_keys
#     for key in keys:
#         matches.append(
#             (list(intersection_cross_pairs[key]), list(centerline_cross_pairs[key]), key))

#     # unmatched centerlines
#     #keys = CXP_keys - IXP_keys
#     # unmatched intersections
#     #keys = IXP_keys - CXP_keys
#     # unmatched centerlines

#     mapping = pd.DataFrame(matches, columns=['intersection_ids', 'centerline_ids', 'sifid_pair'])
#     return mapping

    
# ICpairs = map_intersection_centerline_cross_pairs()

# intersection_counts = ICpairs.intersection_ids.apply(len)
# ICpairs.head()
# #ICpairs length == 18586

# #ICpairs[intersection_counts < 1]
# # -> empty. No such cases


In [384]:
# ICpairs[intersection_counts == 1]
# simple case: case one intersection and its connecting roads.

def map_1_to_many(IXCpairs, centerlines):
    low_match = defaultdict(set)
    hi_match = defaultdict(set)
    matches = IXCpairs[IXCpairs.intersection_ids.apply(len) == 1]

    for intersection in matches.itertuples():
        intersection_id = intersection.intersection_ids[0]
        sifid_1, sifid_2 = intersection.Index
        centerline_matches = centerlines.loc[intersection.centerline_ids]

        for centerline in centerline_matches.itertuples():
            centerline_sifid = centerline.SIFID
            centerline_id = centerline.Index

            if centerline_sifid == sifid_1:
                if centerline.SIFIDLOW == sifid_2:
                    low_match[centerline_id].add(intersection_id)
                elif centerline.SIFIDHI == sifid_2:
                    hi_match[centerline_id].add(intersection_id)
                else:
                    raise ValueError(f"row id : {intersection.Index}")
                
            elif centerline_sifid == sifid_2:
                if centerline.SIFIDLOW == sifid_1:
                    low_match[centerline_id].add(intersection_id)
                elif centerline.SIFIDHI == sifid_1:
                    hi_match[centerline_id].add(intersection_id)
                else:
                    raise ValueError(f"row id : {intersection.Index}")
                
    assert all(len(value) == 1 for value in low_match.values())
    assert all(len(value) == 1 for value in hi_match.values())
    
    #return low_match, hi_match
    return (pd.Series(low_match, name="low_match").apply(lambda x:set.copy(x).pop()),
            pd.Series(hi_match, name="hi_match").apply(lambda x:set.copy(x).pop()))
            

low_match_1, hi_match_1 = map_1_to_many(IXCmatches, centerlines)

In [385]:
# use this information tobuild the intersection network:
low_match_1

display(
centerlines.loc[[13164, 24805, 26558]],
intersections.loc[389674641274662])

lmvc = low_match_1.value_counts()
lmvc[lmvc > 1]

low_match_1[low_match_1 == 389674641274662]

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
13164,S JACKSON ST,3160,3520,5109,"(-85.7461475359, 38.24196786)","(-85.7460293342, 38.2426798605)",MINOR ARTERIAL
24805,S JACKSON ST,3160,3520,657,"(-85.7463969464, 38.2405307151)","(-85.7461475359, 38.24196786)",MINOR ARTERIAL
26558,LAMPTON ST,3520,3160,5427,"(-85.7461475359, 38.24196786)","(-85.7455010614, 38.2419007874)",LOCAL


FST_ROADNAME                       S JACKSON ST
FST_SIFID                                  3160
SEC_ROADNAME                         LAMPTON ST
SEC_SIFID                                  3520
GEOMETRY        (-85.7461426535, 38.2419605705)
Name: 389674641274662, dtype: object

13164    389674641274662
24805    389674641274662
26558    389674641274662
Name: low_match, dtype: int64

In [386]:
#ICpairs[intersection_counts >= 1 ]
# more complex cases
# sifids alone are not sufficient to diambiguate everything 
# will use geometry

# fast euclidean distance between points
euclidean_distance = np.linalg.norm

def pythag(pointA, pointB):
    """Calculate Euclidean distance."""
    return euclidean_distance(np.array(pointA) - pointB)



In [387]:
def map_complex(IXCpairs, intersections, centerlines):
    complex_matches = IXCpairs[IXCpairs.intersection_ids.apply(len) > 1]
    return complex_matches


# went back to easy matches to make sure the intersection network approach would even work
# can we build intersection directy for there here? 
# Rather than making the intermediate centerlines step?

map_complex(IXCmatches, intersections, centerlines)

intersection_ids  \
4976  5908                          [14300767569, 10005800273]   
      5483                        [2574101274724, 91610177636]   
7957  10073                      [174221152281, 5942362230809]   
8646  12316                       [423214159975, 418919192679]   
8315  8538                       [1119183476550, 582312564550]   
...                                                        ...   
6587  15553                 [846696816931636, 846701111898932]   
14940 15575                 [846911567622978, 846928682350372]   
15254 15593                 [847122022987010, 847126317954306]   
15593 15594                 [847130612987714, 847134907955010]   
15495 15596  [847177857758787, 847182152726083, 84718644769...   

                                               centerline_ids  
4976  5908                  [13504, 7109, 84838, 31468, 9233]  
      5483                [12837, 90023, 31054, 28498, 14678]  
7957  10073          [13152, 31204, 549, 19631, 28276, 29783]  
8646  12316           [5218, 3171, 19017, 19343, 25912, 3549]  
8315  8538                         [7992, 13801, 9357, 21943]  
...                                                       ...  
6587  15553                  [171585, 171586, 171590, 171591]  
14940 15575          [173187, 173192, 173193, 173197, 173200]  
15254 15593  [174786, 174788, 174474, 174476, 174478, 174479]  
15593 15594                  [174474, 174475, 174788, 174477]  
15495 15596                  [174792, 163913, 174793, 163917]  

[1075 rows x 2 columns]

In [388]:
mc = centerlines.loc[[13504, 7109, 84838, 31468, 9233]]
geolow_groups = mc.groupby('GEOLOW').groups
geohi_groups = mc.groupby('GEOHI').groups


In [389]:

centerlines.loc[[84838, 31468]]
#(-85.5284187234, 38.2003603491)

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
84838,TUCKER STATION RD,5908,12711,4976,"(-85.5296151016, 38.2036726933)","(-85.5284187234, 38.2003603491)",PRIMARY COLLECTOR
31468,REHL RD,4976,12707,5908,"(-85.531149709, 38.2005045069)","(-85.5284187234, 38.2003603491)",SECONDARY COLLECTOR


In [390]:
intersections.loc[[14300767569, 10005800273]]

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
14300767569,REHL RD,4976,TUCKER STATION RD,5908,"(-85.5284139077, 38.2003530575)"
10005800273,REHL RD,4976,TUCKER STATION RD,5908,"(-85.52814980550001, 38.2003487994)"


In [391]:

def map_many_to_many(IXCpairs, intersections, centerlines):
    low_match = defaultdict(set)
    hi_match = defaultdict(set)
    #distances = pd.DataFrame(columns=['low_distance', 'hi_distance'])

    intersection_points = intersections.GEOMETRY.apply(np.array)

    m2m_matches = IXCpairs[IXCpairs.intersection_ids.apply(len) > 1]

    for row in m2m_matches.itertuples():
        fst_sifid, sec_sifid = row.Index
        intersection_ids = list(row.intersection_ids)
        centerline_data = centerlines.loc[row.centerline_ids]
    
        shared_point_map = defaultdict(set)
        for centerline_id, geolow, geohi in centerline_data[['GEOLOW', 'GEOHI']].itertuples():
            shared_point_map[geolow].add(centerline_id)
            shared_point_map[geohi].add(centerline_id)
        shared_points = shared_point_map.keys()

        # might have over complicated the matching algo here
        # not complicated enough. Still needs some work
        # potential fase matches

        # point_distances = defaultdict(dict)
        # for intersection_id in intersection_ids:
        #     point_1 = intersection_points.at[intersection_id]
        #     for point_2 in point_map.keys():
        #         distance = norm(point_1 - point_2)
        #         point_distances[intersection_id][distance] = point_2

        point_matches = dict()
        for intid, point_1 in intersection_points.loc[intersection_ids].items():
            distances = [(euclidean_distance(point_1 - point_2), point_2) for point_2 in shared_points]
            _, point_2 = min(distances, key=itemgetter(0))
            point_matches[intid] = list(shared_point_map[point_2])
            
        for intersection_id, centerline_ids in point_matches.items():
            for centerline in centerline_data.loc[centerline_ids].itertuples():
                centerline_id = centerline.Index
                centerline_sifid = centerline.SIFID
                if centerline_sifid == fst_sifid:
                    if centerline.SIFIDLOW == sec_sifid:
                        low_match[centerline_id].add(intersection_id)
                    elif centerline.SIFIDHI == sec_sifid:
                        hi_match[centerline_id].add(intersection_id)
                    else:
                        raise ValueError("bad centerline for intersection: "+
                            f"intid {intersection_id} centerline {centerline_id}")
                
                elif centerline_sifid == sec_sifid:
                    if centerline.SIFIDLOW == fst_sifid:
                        low_match[centerline_id].add(intersection_id)
                    elif centerline.SIFIDHI == fst_sifid:
                        hi_match[centerline_id].add(intersection_id)
                    else:
                        raise ValueError("bad centerline for intersection: "+
                            f"intid {intersection_id} centerline {centerline_id}")
                else:
                    raise ValueError("bad centerline for intersection: "+
                        f"intid {intersection_id} centerline {centerline_id}")
                
    return pd.Series(low_match, name='low_match'), pd.Series(hi_match, name='hi_match')
            
low_match_m2m, hi_match_m2m = map_many_to_many(IXCmatches, intersections, centerlines)


In [392]:
# some centerline ids have more than one match
low_match_m2m[low_match_m2m.apply(len) > 1]


7109              {10005800273, 14300767569}
31054           {91610177636, 2574101274724}
7992           {582312564550, 1119183476550}
21943          {582312564550, 1119183476550}
25745         {1379545420614, 1521279341382}
                         ...                
173192    {846911567622978, 846928682350372}
174475    {847130612987714, 847134907955010}
174477    {847130612987714, 847134907955010}
174793    {847186447693379, 847177857758787}
163913    {847186447693379, 847182152726083}
Name: low_match, Length: 1092, dtype: object

In [393]:
low_match_m2m[low_match_m2m.apply(len) == 2]
# 1312 have 2 matches
# removing ramps and expressways removed about 200 records
# new total: 1089

# a lot of these have SIFIDHI == SIFIDLOW
    # 1010 <- about 200 removed by eliminating expressways and ramps from centerlines

mx2 = low_match_m2m[low_match_m2m.apply(len) == 2]
mx2
mx22 = centerlines.loc[mx2.index]
mismatch22 = mx22[(mx22.SIFIDHI != mx22.SIFIDLOW)]

# about 79 do not

mismatch22


,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
67216,SWEET GUM LN,6310,2017,6518,"(-85.5154532259, 38.3069827797)","(-85.5158693412, 38.3074752761)",LOCAL
25217,TAYLORSVILLE RD,5769,8762,2332,"(-85.649619226, 38.2213821789)","(-85.6462567063, 38.221106526)",MAJOR ARTERIAL
27672,BROWNSBORO RD,716,8763,12531,"(-85.5717353492, 38.3098244748)","(-85.5702122827, 38.311002072)",MINOR ARTERIAL
20966,RUDY LN,5145,3268,8691,"(-85.6389157503, 38.2791572951)","(-85.6399130723, 38.2804096818)",LOCAL
29197,GIRARD DR,12920,2494,2003,"(-85.6170743735, 38.2660117383)","(-85.6178083177, 38.2668345905)",LOCAL
...,...,...,...,...,...,...,...
174155,FARRIER DR,15476,15138,1719,"(-85.4496235174, 38.2653509338)","(-85.448797353, 38.2672949025)",LOCAL
162317,W MANSLICK RD,3898,15483,3368,"(-85.7932912759, 38.1183029083)","(-85.7917596513, 38.1180548734)",PRIMARY COLLECTOR
162640,STAR HILL DR,15484,15486,2132,"(-85.6170285085, 38.1594844225)","(-85.6170559663, 38.1588304429)",LOCAL
167433,OXMOOR WOODS PKY,6806,1785,6883,"(-85.6073745926, 38.2435979747)","(-85.6063790349, 38.2434420861)",LOCAL


In [394]:
mx2.loc[67216]
intersections.loc[[7095824117768, 721281507401747]]

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
7095824117768,ELM DR,2017,WALNUT ALY,6310,"(-85.515871903, 38.307477049)"
721281507401747,SWEET GUM LN,6310,BLACK GUM LN,2017,"(-85.5154484061, 38.3069754671)"


In [395]:
def m2m_match_2(matches, intersections, centerlines):
    low_match = defaultdict(set)
    hi_match = defaultdict(set)
    for cl_index, intersection_ids in matches[matches.apply(len) == 2].items():
        centerline = centerlines.loc[cl_index]
        if centerline.SIFIDLOW == centerline.SIFIDHI:
            cl_geolow = np.array(centerline.GEOLOW)
            cl_geohi = np.array(centerline.GEOHI)

            int1_id, int2_id = intersection_ids
            int1_geo = intersections.at[int1_id, 'GEOMETRY']
            int2_geo = intersections.at[int2_id, 'GEOMETRY']

            geolow_int1 = euclidean_distance(cl_geolow - int1_geo)
            geohi_int1 = euclidean_distance(cl_geohi - int1_geo)

            geolow_int2 = euclidean_distance(cl_geolow - int2_geo)
            geohi_int2 = euclidean_distance(cl_geohi - int2_geo)

            if geolow_int1 < geolow_int2:
                low_match[cl_index].add(int1_id)
            elif geolow_int2 < geolow_int1:
                low_match[cl_index].add(int2_id)

            if geohi_int1 < geohi_int2:
                hi_match[cl_index].add(int1_id)
            elif geohi_int2 < geohi_int1:
                hi_match[cl_index].add(int2_id)
        
        else: 
            ...
            
    return pd.Series(low_match, name='low_match'), pd.Series(hi_match, name='hi_match')
        


low, hi = m2m_match_2(low_match_m2m, intersections, centerlines)        

low.apply(lambda x:len(x) == 1).all() and hi.apply(lambda x:len(x) == 1).all()
# --> True Good news. We found one match for each centerline_id
#. not sure all these matches are right, though

np.True_

In [396]:
for cindex, imatches in low_match_m2m.loc[mismatch22.index].items():
    display(mismatch22.loc[cindex])
    display(intersections.loc[list(imatches)])

ROADNAME                         SWEET GUM LN
SIFID                                    6310
SIFIDLOW                                 2017
SIFIDHI                                  6518
GEOLOW        (-85.5154532259, 38.3069827797)
GEOHI         (-85.5158693412, 38.3074752761)
CORE_CLASS                              LOCAL
Name: 67216, dtype: object

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
7095824117768,ELM DR,2017,WALNUT ALY,6310,"(-85.515871903, 38.307477049)"
721281507401747,SWEET GUM LN,6310,BLACK GUM LN,2017,"(-85.5154484061, 38.3069754671)"


ROADNAME                     TAYLORSVILLE RD
SIFID                                   5769
SIFIDLOW                                8762
SIFIDHI                                 2332
GEOLOW        (-85.649619226, 38.2213821789)
GEOHI         (-85.6462567063, 38.221106526)
CORE_CLASS                    MAJOR ARTERIAL
Name: 25217, dtype: object

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
20303981091221,TAYLORSVILLE RD,5769,I 264 RAMP,8762,"(-85.6496143734, 38.2213748889)"
20355520698773,TAYLORSVILLE RD,5769,I 264 RAMP,8762,"(-85.6462518546, 38.2210992359)"


ROADNAME                        BROWNSBORO RD
SIFID                                     716
SIFIDLOW                                 8763
SIFIDHI                                 12531
GEOLOW        (-85.5717353492, 38.3098244748)
GEOHI          (-85.5702122827, 38.311002072)
CORE_CLASS                     MINOR ARTERIAL
Name: 27672, dtype: object

...

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
844802722952754,CHELSEA GARDENS CIR,15486,STAR HILL DR,15484,"(-85.6170511279, 38.1588231634)"
844811312887346,CHELSEA GARDENS CIR,15486,STAR HILL DR,15484,"(-85.61702367, 38.1594771429)"


ROADNAME                     OXMOOR WOODS PKY
SIFID                                    6806
SIFIDLOW                                 1785
SIFIDHI                                  6883
GEOLOW        (-85.6073745926, 38.2435979747)
GEOHI         (-85.6063790349, 38.2434420861)
CORE_CLASS                              LOCAL
Name: 167433, dtype: object

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
845612314007433,OXMOOR WOODS PKY,6806,CHRISTIAN WAY,1785,"(-85.6073697506, 38.2435906784)"
845603724072841,OXMOOR WOODS PKY,6806,CHRISTIAN WAY,1785,"(-85.6063741932, 38.2434347899)"


ROADNAME                          WELLHEAD DR
SIFID                                   15596
SIFIDLOW                                15495
SIFIDHI                                  8594
GEOLOW        (-85.6602212563, 38.1331011682)
GEOHI         (-85.6608151736, 38.1316663553)
CORE_CLASS                              LOCAL
Name: 163913, dtype: object

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
847186447693379,WELLHEAD DR,15596,SHEARLING DR,15495,"(-85.6602164072, 38.1330938957)"
847182152726083,WELLHEAD DR,15596,SHEARLING DR,15495,"(-85.6608103243, 38.131659083)"


In [397]:

cl, ix =88673, {334354632500584, 726302145904921}

display(centerlines.loc[cl])
display(intersections.loc[list(ix)])

ROADNAME                          ANDERSON ST
SIFID                                     110
SIFIDLOW                                 5924
SIFIDHI                                  5930
GEOLOW        (-85.7849841918, 38.2484300342)
GEOHI          (-85.7866493992, 38.248495505)
CORE_CLASS                              LOCAL
Name: 88673, dtype: object

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
334354632500584,ANDERSON ST,110,S 20TH ST,5924,"(-85.7849792976, 38.2484227453)"
726302145904921,S 20TH ST,5924,ANDERSON ST,110,"(-85.7850776812, 38.2478758959)"


In [398]:


# 5 have 3 matches

# removing ramps from centerlines removed 2 of these

# 11577    {318119662229912, 318128252164504, 31811536726...
# Arthur St. and I 65 RAMP

# 79773    {731803473713560, 844908543362584, 84487847859...
# I 65 ramp

# 80073    {844438991051160, 731813464524312, 73181635861...
# I 65 ramp

# 31660    {582161378284545, 582191443055617, 61834218278...

#582161378284545	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.8735529704, 38.0723544119)
#582191443055617	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.8741780941, 38.0721878176)
#618342182786049	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.873007987, 38.0733107806)

# 19920    {731744745003415, 722828392896919, 72287563753...
# I 65 RAMP



# 1 has 4 matches -> 
# removing EXPRESSWAYS from centerlines dealt with this one.

# 79772 : {731803473713560, 731813464524312, 844878478591512, 844908543362584}
# I 65 NORTH x I 65 RAMP
# no more from there 



In [399]:
# #len(matches)
# #count = set()
# #for cx in ICpairs.centerlines.values
# #    count.update(cx)
# ###len(count)
# #len(matches)
# matches
# #matches.low_match.isna().combine(matches.hi_match.isna(), lambda x,y:x & y)

# # some centerlines have both himatch and low match 
# mi = matches[['low_match', 'hi_match']].isna()#.any(axis=1)
# #.apply(lambda x:x.isna()).sum(axis=1).value_counts()
# matches[~mi.any(axis=1)]
# #matches[mi]
# mi.all(axis=1).any() # no cases where there are no matches, always 1 or 2
# #centerlines.loc[mi.index]

In [400]:
# potential problematic numbers

#set(hi_match.keys()).intersection(hi_match2.keys()) # empty : good news
#  14288: ('start', 360168983520312, 8.767631726955189e-06),

#display(
##centerlines.loc[14288],
#intersections.loc[[360168983520312, 360748804105272]]
#

#centerlines.loc[[52803, 20860]]
#centerlines[centerlines.SIFIDLOW == 8594]
#centerlines.loc[[ 1498,  2541,  2610,  3823,  3882,  5039,  6722,  7769,  8692, 11961,
#       14175, 15423, 19704, 23933, 24872, 27737, 29258, 30870]] # I 265 RAMP.
#intersections.loc[847265632743817]
#intersections.loc[722828392896919]
#centerlines.loc[19920]
#ICpairs.loc[6146]

In [401]:
#cell label 1 #!!! takes a long time to run ;0)
#centerlines_by_SIFID = centerlines.groupby(['SIFID']).groups
#intersections[['FST_SIFID', 'SEC_SIFID']]

#intersection_roadways = pd.DataFrame(columns=("fst_roadways", "sec_roadways"))

#def new():
#    return {'intersection_1':set(), 'intersection_2':set()}
#roadway_intersections = defaultdict(new)

# for intersection in intersections[['FST_SIFID', 'SEC_SIFID']].itertuples():
#     intersection_id = intersection.Index
#     sifid_1 = intersection.FST_SIFID
#     sifid_2 = intersection.SEC_SIFID

#     cl_id_1 = centerlines_by_SIFID.get(sifid_1, None)
#     cl_id_2 = centerlines_by_SIFID.get(sifid_2, None)

#     if cl_id_1 is not None:
#         fst_rwy = centerlines.loc[cl_id_1]
#         int_fst_rwy = fst_rwy[(fst_rwy.SIFIDLOW == sifid_2) | (fst_rwy.SIFIDHI == sifid_2)].index
#         if not int_fst_rwy.empty:
#             intersection_roadways.at[intersection_id, 'fst_roadways'] = int_fst_rwy.tolist()
#             for roadway_id in int_fst_rwy:
#                 roadway_intersections[roadway_id]['intersection_1'].add(intersection_id)

#     if cl_id_2 is not None:
#         sec_rwy = centerlines.loc[cl_id_2]
#         int_sec_rwy = sec_rwy[(sec_rwy.SIFIDLOW == sifid_1) | (sec_rwy.SIFIDHI == sifid_1)].index
#         if not int_sec_rwy.empty:
#             intersection_roadways.at[intersection_id, 'sec_roadways'] = int_sec_rwy.tolist()
#             for roadway_id in int_sec_rwy:
#                 roadway_intersections[roadway_id]['intersection_2'].add(intersection_id)


In [402]:
# # earlier version of cell label 1
# cxsifid = centerlines_by_SIFID

# #intx = intersections.sample()
# intx = intersections.loc[5710837346]
# fst = intx.FST_SIFID.item()
# sec = intx.SEC_SIFID.item()

# fst_rwy = centerlines.loc[cxsifid[fst]]
# sec_rwy = centerlines.loc[cxsifid[sec]]

# int_fst_rwy = fst_rwy[(fst_rwy.SIFIDLOW == sec) | (fst_rwy.SIFIDHI == sec)]
# int_sec_rwy = sec_rwy[(sec_rwy.SIFIDLOW == fst) | (sec_rwy.SIFIDHI == fst)]

# display("intersection", intx)
# display('roadways', int_fst_rwy, int_sec_rwy)

In [403]:
# low_connect = centerlines.groupby(["SIFID", "SIFIDLOW"]).groups
# hi_connect = centerlines.groupby(['SIFID', 'SIFIDHI']).groups

# # data = pd.DataFrame(columns=["fst_match", "sec_match"])

# # c2i = defaultdict(set)

# # for intid, (sifid_1, sifid_2) in intersections[['FST_SIFID', 'SEC_SIFID']].iterrows():
# #     pair = (sifid_1, sifid_2)
# #     fst_match = hi_connect.get(pair, None)
# #     if fst_match is None:
# #         fst_match = low_connect.get(pair, None)
# #     if fst_match is not None:
# #         data.at[intid, 'fst_match'] = fst_match
# #         for centerline_id in fst_match:
# #             c2i[centerline_id].add(intid)


# #     pair = (sifid_2, sifid_1)
# #     sec_match = hi_connect.get(pair, None)
# #     if sec_match is None:
# #         sec_match = low_connect.get(pair, None)
# #     if sec_match is not None:
# #         data.at[intid, 'sec_match'] = sec_match
# #         for centerline_id in sec_match:
# #             c2i[centerline_id].add(intid)

# # #c2i = pd.DataFrame.from_records(
# #     #list(c2i.items()), columns=['centerline_id', 'intersection_id']).set_index('centerline_id')

# # #centerlines[['SIFIDHI', 'SIFIDLOW']].notna().any(axis=1).all() # good. all centerlines have either sifidlow or sifidhi: 
# # # no isolated roads, which makes sense. No reason to have a city road that doesn't connect to the road system

# # not_found = data[(data.fst_match.isna()) & (data.sec_match.isna())].index

# # intersections.loc[not_found]

# # data[data.notna().any(axis=1)]

# # data

In [404]:
# roadways = centerlines.groupby('SIFID').groups
# intxn_by_fst_sifid = intersections.groupby('FST_SIFID').groups
# # intxn_by_sec_sifid = intersections.groupby('SEC_SIFID') # probably not necessary
#     # ... since all records will be accessed by iterating over the fst_sifid groupby

# #intersections.FST_SIFID.hasnans # == False this is good

# def map_intersection_ids_to_centerline_ids(intersection_sifid_groups=intxn_by_fst_sifid, centerline_sifid_groups=roadways):
#     found = dict()
#     notfound = list()
#     for sifid_1 in intersection_sifid_groups.keys():
#         roadway_ids = centerline_sifid_groups.get(sifid_1, None)
#         if roadway_ids is not None:
#             found[sifid_1] = roadway_ids # first sifid match to centerlines id
#         else:
#             notfound.append(sifid_1)
#     return found, notfound

# def map_intersections_to_centerlines(intersections=intersections, centerlines=centerlines):
#     mapping = pd.DataFrame(columns=['full_match', 'fst_match_only', 'sec_match_only'])
#     full_matches = pd.Series(name='full_match', dtype="O")
#     roadways = centerlines.groupby('SIFID').groups
#     intersections_by_first_sifid = intersections.groupby('FST_SIFID').groups
#     notfound = list()

#     for sifid_1, intersection_ids in intersections_by_first_sifid.items():
#         centerline_ids = roadways.get(sifid_1, None)
#         if centerline_ids is None:
#             notfound.extend(intersection_ids)
#         else:
#             fst_match = centerlines.loc[centerline_ids]
#             for intersection_id in intersection_ids:
#                 sifid_2 = intersections.at[intersection_id, 'SEC_SIFID']
#                 full_match = fst_match[(fst_match.SIFIDLOW == sifid_2) | (fst_match.SIFIDHI == sifid_2)]
#                 if full_match.empty:
#                     mapping.at[intersection_id, 'fst_match_only'] = True
#                 else:
#                     full_match = full_match.index.tolist()
#                     #print(full_match)
#                     full_matches.at[intersection_id] = full_match
        
#     if notfound:
#         centerline_sifids = roadways.keys()
#         for intersection_id, sifid_2 in intersections.loc[notfound]['SEC_SIFID'].items():
#             if sifid_2 in centerline_sifids:
#                 mapping.at[intersection_id, 'sec_match_only'] = True

#     return pd.concat((full_matches, mapping))


# mapping = map_intersections_to_centerlines()


In [405]:
#mapping.isna().apply(any, axis=1).all()
# test that each row has at least one value filled -> Yes
#intersections.index.difference(mapping.index) # == Index([693211604576105], dtype='int64')

#display(intersections.loc[693211604576105]) -> 13554, 13555 fst, sec ids

#centerlines[centerlines.SIFID == 13555] # nope, nor 13554 
# probably just ignore this.

#mapping[mapping.full_match.notna()]
#mapping

In [406]:
# mapping = pd.DataFrame(columns=['full_match', 'fst_match_only', 'sec_match_only'])
# roadways = centerlines.groupby('SIFID').groups
# intersections_by_first_sifid = intersections.groupby('FST_SIFID').groups
# notfound = list()

# for sifid_1, intersection_ids in intersections_by_first_sifid.items():
#     centerline_ids = roadways.get(sifid_1, None)
#     if centerline_ids is None:
#         notfound.extend(intersection_ids)
#     else:
#         fst_match = centerlines.loc[centerline_ids]
#         #print(fst_match)
#         for intersection_id in intersection_ids:
#             sifid_2 = intersections.at[intersection_id, 'SEC_SIFID']
#             full_match = fst_match[(fst_match.SIFIDLOW == sifid_2) | (fst_match.SIFIDHI == sifid_2)]



In [407]:
centerlines[centerlines.SIFID==1178]
notfound = [624,626,689,903,2053,122932,139877,161371,161372,168074]

intersections[intersections.FST_SIFID == 1178]
nf1 = intersections[intersections.FST_SIFID.isin(notfound)]
nf2 = centerlines[centerlines.SIFID.isin(nf1.SEC_SIFID.values)] # looks to be all interstate ramps
# it makes sense that these would not be in the centerline data b/c I probably stripped them out at a some point
# you can't/shouldn't ride a bicycle on the ramps/interstate!

noramp = nf2[nf2.CORE_CLASS != "INTERSTATE RAMP"] # 205 ramps. We don't need these roadways.
#display(noramp)
#nf.groupby('CORE_CLASS').count()

#nf.groupby('SIFID').groups
#sifid_twos = intersections[intersections.FST_SIFID.isin(notfound)].SEC_SIFID.values#.groupby('SEC_SIFID').groups
#for sifid_2 in sifid_twos:
#    i = centerlines[centerlines.SIFID == sifid_2].index.tolist()
#    if not i:
#        print(sifid_2)

display(nf1, nf2)

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
25314647488608,BRITTANY VALLEY RD,689,LIME KILN LN,3636,"(-85.6373935734, 38.2990193474)"
25533690815846,BRITTANY VALLEY RD,689,GLENVIEW AVE,2525,"(-85.6434704537, 38.2964091193)"
285078996590724,COFFEE TREE LN,2053,COFFEE TREE PL,2101,"(-85.5369278673, 38.1819592614)"
374478301589619,BOWLES AVE,626,WEBSTER ST,6361,"(-85.7267989454, 38.2571141241)"
374942158031000,BOWLES AVE,626,CABEL ST,875,"(-85.7283190618, 38.2564620304)"
414945534616712,CANDOR AVE,903,GARRS LN,2460,"(-85.8163622937, 38.1897925887)"
422723720394881,CANDOR AVE,903,LINHERK AVE,3654,"(-85.8169944951, 38.1869962274)"
432752417703459,BOWIE CT,624,BOWIE DR,625,"(-85.7026014005, 38.1465724184)"


,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
3034,LIME KILN LN,3636,3536,5040,"(-85.6447336957, 38.3128469693)","(-85.6455583868, 38.313566956)",PRIMARY COLLECTOR
3125,WEBSTER ST,6361,13416,626,"(-85.726472992, 38.2566529689)","(-85.7268038233, 38.2571214174)",LOCAL
3474,GARRS LN,2460,903,4265,"(-85.8163671926, 38.1897998649)","(-85.8174152907, 38.1898710161)",LOCAL
4249,GLENVIEW AVE,2525,1696,8594,"(-85.6420143435, 38.2943749671)","(-85.6421980563, 38.2945737706)",LOCAL
4476,BOWIE DR,625,624,13326,"(-85.702606263, 38.1465796916)","(-85.7016940176, 38.1468951256)",LOCAL
...,...,...,...,...,...,...,...
31463,GLENVIEW AVE,2525,2626,12896,"(-85.6407907198, 38.2928495414)","(-85.6412544794, 38.2934332755)",LOCAL
32265,COFFEE TREE PL,2101,8594,2053,"(-85.5379593471, 38.1819082319)","(-85.5369458175, 38.1819669912)",LOCAL
32380,GARRS LN,2460,3661,4085,"(-85.8131860153, 38.1896095928)","(-85.8150640388, 38.1897210539)",LOCAL
32411,LIME KILN LN,3636,1197,2517,"(-85.634665912, 38.2955398449)","(-85.6357147464, 38.2968583401)",PRIMARY COLLECTOR


In [408]:
# # # This is where Rehl Rd intersects Tucker station



#ee = centerlines[A][["SIFIDLOW", "SIFIDHI"]].copy()
# 5908 another test

#ee['next'] = pd.Series(dtype='int64')
#ee['previouos'] = pd.Series(dtype='int64')

#indexes = set(ee.index)

#while indexes:
#    row = ee.loc[indexes.pop()]
#    if row.SIFIDHI == row.SIFIDLOW:
#        continue # handle loops: 

""" SIFIDLOW	SIFIDHI	next	previouos
7109	5908	5908	NaN	NaN # next previous should be 12697?
9233	5908	12697	NaN	NaN
...
52804	12697	4674	NaN	NaN

This is where Rehl Rd intersects Tucker station

       | Tucker Station
       |
--Rehl--====Rehl====----Rehl--
                  |
                  | Tucker Station

This makes it too annoying to connect the segments using only SIFIDS. Lots of unusual cases.
We have exact geometry data, just use that
code for this already in centerline_data notebook """

' SIFIDLOW\tSIFIDHI\tnext\tpreviouos\n7109\t5908\t5908\tNaN\tNaN # next previous should be 12697?\n9233\t5908\t12697\tNaN\tNaN\n...\n52804\t12697\t4674\tNaN\tNaN\n\nThis is where Rehl Rd intersects Tucker station\n\n       | Tucker Station\n       |\n--Rehl--====Rehl====----Rehl--\n                  |\n                  | Tucker Station\n\nThis makes it too annoying to connect the segments using only SIFIDS. Lots of unusual cases.\nWe have exact geometry data, just use that\ncode for this already in centerline_data notebook '